## Simulate Medium absolute deviation about the median:

In [2]:
using Random, Distributions, Statistics
using Convex, MosekTools, LinearAlgebra

# ---------- mixture grid + fit (unchanged) ----------
function default_grid_for_U(U::Vector{Float64}; n_mu=20, n_sigma=50)
    qlo, qhi = quantile(U, (0.0001, 0.9999))
    pad = 0.2 * (qhi - qlo)
    mus = collect(range(qlo - pad, qhi + pad; length=n_mu))

    sU = std(U)
    σ_lo = max(1e-3, 0.05 * sU)
    σ_hi = max(σ_lo * 2, 2.0 * sU)
    sigmas = exp.(range(log(σ_lo), log(σ_hi); length=n_sigma))

    return mus, sigmas
end

function fit_fixedgrid_gaussian_mixture_mosek(U::Vector{Float64};
    mus::Vector{Float64},
    sigmas::Vector{Float64},
    verbose::Bool=false
)
    N = length(U)

    mu_list = Float64[]
    sig_list = Float64[]
    for μ in mus, σ in sigmas
        push!(mu_list, μ)
        push!(sig_list, σ)
    end
    K = length(mu_list)

    logφ = Matrix{Float64}(undef, N, K)
    @inbounds for j in 1:K
        dj = Normal(mu_list[j], sig_list[j])
        for i in 1:N
            logφ[i, j] = logpdf(dj, U[i])
        end
    end

    m = vec(maximum(logφ, dims=2))
    A = exp.(clamp.(logφ .- m, -745.0, 0.0))

    π = Variable(K)
    constraints = [π >= 0, sum(π) == 1]
    obj = sum(m) + sum(log(A * π))
    problem = maximize(obj, constraints)

    solve!(problem, Mosek.Optimizer; silent=!verbose)

    π_hat = vec(evaluate(π))
    π_hat = max.(π_hat, 0.0)
    π_hat ./= sum(π_hat)

    comps = (mu = mu_list, sigma = sig_list)
    return π_hat, comps
end

# ---------- NEW: median + MAD about median ----------
@inline mad_median(x) = median(abs.(x .- median(x)))   # MAD about the median

"""
Simulate B datasets of size n from N(0,1), return:
  T  = median(x)
  D  = MAD about median = median(|x - median(x)|)
  U  = log(D)
"""
function simulate_median_MADmed_U(n::Int, B::Int; rng=Random.default_rng())
    T = Vector{Float64}(undef, B)   # median
    D = Vector{Float64}(undef, B)   # MAD about median
    x = Vector{Float64}(undef, n)
    dist = Normal(0,1)

    @inbounds for b in 1:B
        rand!(rng, dist, x)
        med = median(x)
        T[b] = med
        D[b] = median(abs.(x .- med))
    end

    U = log.(D .+ 1e-300)
    return T, D, U
end

simulate_median_MADmed_U

## n=3

In [5]:


# ---------- example run ----------
rng = MersenneTwister(3)
n = 3
B = 600_000

T, D, U = simulate_median_MADmed_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2354.69 seconds, 150.475 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 2.78            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 1.0


In [8]:
comps

(mu = [-11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437, -11.884326515428437  …  3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825, 3.0830455513220825], sigma = [0.05904155370201365, 0.06365798381460101, 0.06863537033243412, 0.07400193625971979, 0.07978811134351488, 0.08602670461786688, 0.09275309043909627, 0.10000540906708622, 0.10782478292992492, 0.1162555497981742  …  1.1993939453555473, 1.2931739694067343, 1.3942866075211302, 1.5033051931942623, 1.6208478885863427, 1.7475811896535607, 1.884223705343875, 2.0315502322862664, 2.1903961480779492, 2.361662148080546])

In [6]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.1_MADmedian(n=3).jld2" π_hat comps n B


## n=4

In [1]:


# ---------- example run ----------
rng = MersenneTwister(3)
n = 4
B = 600_000

T, D, U = simulate_median_MADmed_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2459.53 seconds, 151.097 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 1.64            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 0.9999999999999999


In [7]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.1_MADmedian(n=4).jld2" π_hat comps n B


## n=5

In [3]:

# ---------- example run ----------
rng = MersenneTwister(3)
n = 5
B = 600_000

T, D, U = simulate_median_MADmed_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2351.38 seconds, 151.097 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 1.54            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 0.9999999999999998


In [4]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.1_MADmedian(n=5).jld2" π_hat comps n B


## n=6

In [2]:
# ---------- example run ----------
rng = MersenneTwister(3)
n = 6
B = 600_000

T, D, U = simulate_median_MADmed_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2500.3 seconds, 151.097 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 1.23            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 1.0


In [3]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.1_MADmedian(n=6).jld2" π_hat comps n B


## n=8

In [2]:
# ---------- example run ----------
rng = MersenneTwister(3)
n = 8
B = 600_000

T, D, U = simulate_median_MADmed_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2431.1 seconds, 151.097 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 1.68            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 1.0


In [4]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.1_MADmedian(n=8).jld2" π_hat comps n B


## n=10

In [2]:
# ---------- example run ----------
rng = MersenneTwister(3)
n = 10
B = 600_000

T, D, U = simulate_median_MADmed_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2422.71 seconds, 151.097 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 1.69            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 0.9999999999999998


In [3]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.1_MADmedian(n=10).jld2" π_hat comps n B
